# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** OpenCode exited with status 0 without a valid queue-result.json. Last captured output: [2026-10-09T01:15:27.976Z] ! permission requested: external_directory (/*); auto-rejecting | [2026-10-09T01:15:28.001Z] ✗ for R in "items=1000-1999" "1000-1999"; do echo "== Range: $R"; curl -sS --max-time 30 -H "Range: $R" "https://data.mendeley.com/public-api/datasets/3xd9n945v8/files?version=1&folder_id=9685434a-9bdd-4088-8e71-f | [2026-10-09T01:15:28.001Z] Error: The user rejected permission to use this specific tool call. | Required queue-result.json is unavailable or invalid: FileNotFoundError

Attempt status: failed. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: Optimizing Support Vector Machine for Avocado Ripeness Classification Using Moth Flame Optimization

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-845668b5bebd667d367e2f37f92142ad)


# MFO-SVM Hass Avocado Ripeness Classification — hands-on reproduction

**Paper:** Kemal Crisannaufal & Wikky Fawwaz Al Maki, *"Optimizing Support Vector Machine for Avocado Ripeness Classification Using Moth Flame Optimization"*, JEEEMI Vol. 7 No. 2 (April 2025), pp. 330–340, DOI [10.35882/jeeemi.v7i2.652](https://doi.org/10.35882/jeeemi.v7i2.652) (CC BY-SA 4.0).

**Author code:** `github.com/kemalcrisannaufal/MFO-SVM-for-Avocado-Ripeness-Classification` @ commit `2d6ab366747a0ae51f394e8789c5106c202fbdc6` (no LICENSE file; reused here for reproducible education with attribution).

**Dataset:** ["'Hass' Avocado Ripening Photographic Dataset"](https://data.mendeley.com/datasets/3xd9n945v8/1) (Mendeley Data, DOI 10.17632/3xd9n945v8.1, CC BY 4.0): 14,710 JPGs at 800×800 plus an XLSX sheet mapping each filename to a ripeness index (1–5).

**What this notebook does (ADAPTED PYTHON DEMONSTRATION).** The paper's full pipeline uses all 14,710 images, geometric augmentation up to 2,500 images/class, 8,897 hand-crafted features (HSV 701 + GLCM 96 + HOG 8,100), PCA to 150 components, and MFO (50 moths × 15 iterations) to tune an RBF-SVM, reporting 82.55% test accuracy vs 81.60% for the default RBF baseline. Running the *full* augmented pipeline (≈12,500 training images × 8,897 features) is beyond a free Colab session, so this notebook runs a **deterministic stratified subset (60 images/class, 300 total)** with the **author's exact feature extraction, split logic, scaler, PCA and MFO settings**, but **no augmentation**. Absolute accuracies will therefore differ from the paper's numbers; the scientific comparison (does MFO-tuned RBF beat the default RBF baseline on the same subset?) is preserved.

*このノートブックは論文のパイプラインをCPUだけで実行できる規模に縮小した「ADAPTED PYTHON DEMONSTRATION」です。著者コードの特徴量抽出（HSV+GLCM+HOG=8,897次元）、70:15:15の層化分割、StandardScaler、PCA(150)、MFO設定（個体数50・15反復）は著者コードの正確な実装に従いますが、データはクラスあたり60枚・計300枚の決定論的サブセットで、幾何学的データ拡張は省略しています。そのため絶対精度は論文値（MFO-RBF 82.55%、ベースライン 81.60%）とは異なります。*

**Runtime:** CPU only — the whole pipeline is scikit-learn / scikit-image / OpenCV; no GPU is used by the method.

## 1. Setup

**Purpose /期待結果:** Install `niapy==2.5.2` (the MFO library the author's `MFO_RBF.ipynb` uses via `pip install niapy`) with `--no-deps`, because newer niapy wheels force numpy/pandas upgrades that conflict with the preinstalled Colab stack (verified in a diagnostic probe: `niapy 2.5.2` imports cleanly on numpy 2.x; 2.7.1 forces numpy 2.5/pandas 3 conflicts). All other dependencies (numpy, scikit-learn, scikit-image, OpenCV, pandas, matplotlib, seaborn, openpyxl) are preinstalled on Colab.

In [ ]:
import subprocess, sys, importlib.metadata as im
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "niapy==2.5.2"],
                   capture_output=True, text=True)
print("pip rc:", r.returncode, (r.stderr or "")[-300:])
import niapy, numpy, sklearn, cv2, skimage, pandas
print("niapy", niapy.__version__ if hasattr(niapy, "__version__") else im.version("niapy"))
print("numpy", numpy.__version__, "| sklearn", sklearn.__version__,
      "| cv2", cv2.__version__, "| skimage", skimage.__version__, "| pandas", pandas.__version__)
from niapy.task import Task, OptimizationType
from niapy.problems import Problem
from niapy.algorithms.basic import MothFlameOptimizer
print("niapy MFO imports OK")

## 2. Download the label sheet (XLSX) from Mendeley

**Purpose /期待結果:** Fetch the dataset's root XLSX (`Avocado Ripening Dataset.xlsx`, 633,351 bytes) and verify its SHA-256 against the value recorded from the Mendeley files API. Mendeley's data host sits behind a bot challenge that rejects plain Python `requests` (HTTP 403 "Just a moment…" page), so this notebook uses the system `curl` client for every Mendeley HTTP call — a diagnostic probe verified `curl` succeeds where `requests` fails, with or without a browser User-Agent. Expected result: the printed SHA-256 matches `f8abeaba…b51a7` and 14,722 rows load.

In [ ]:
import subprocess, hashlib, re
XLSX_URL = f"https://data.mendeley.com/public-files/datasets/3xd9n945v8/files/0de8d406-a0cc-41d1-820a-9a4de580e114/file_downloaded"
EXPECTED_SHA256 = "f8abeaba6eedf67869907b3a71ee132cd89b42bb3dcce6b25ddb22e6269b51a7"

def mendeley_curl(url, out_path):
    p = subprocess.run(["curl", "-sS", "-L", "--fail", "--max-time", "120",
                        "-o", out_path, "-w", "%{http_code} %{content_type} %{size_download}", url],
                       capture_output=True, text=True)
    print(out_path, "->", p.stdout)
    if p.returncode != 0:
        raise RuntimeError(f"curl failed for {url}: {p.stderr[:200]}")

mendeley_curl(XLSX_URL, "/content/avocado_ripening_dataset.xlsx")
sha = hashlib.sha256(open("/content/avocado_ripening_dataset.xlsx", "rb").read()).hexdigest()
print("sha256:", sha)
assert sha == EXPECTED_SHA256, "XLSX checksum mismatch — refusing to continue with unverified bytes."
print("XLSX verified.")

### 2.1 Read the label table and enumerate the dataset files

**Purpose /期待結果:** The XLSX has 14,722 rows with columns `File Name` (stem, no extension) and `Ripening Index Classification` (1–5: 1=Underripe, 2=Breaking, 3=Ripe first stage, 4=Ripe second stage, 5=Overripe — the five classes of paper TABLE I). We then page the public Mendeley **files API** (metadata only, 1,000 entries per page, 15 pages) to enumerate every JPG's download UUID. Expected result: ≈14,710 JPG entries whose stems mostly intersect the XLSX label table.

In [ ]:
import json, subprocess
import pandas as pd

df = pd.read_excel("/content/avocado_ripening_dataset.xlsx")
print(df.shape, list(df.columns))
print(df["Ripening Index Classification"].value_counts().sort_index())

FOLDER_ID = "9685434a-9bdd-4088-8e71-f025b4c91bff"
entries = []
for offset in range(0, 20000, 1000):
    u = (f"https://data.mendeley.com/public-api/datasets/3xd9n945v8/files"
         f"?version=1&folder_id={FOLDER_ID}&offset={offset}")
    p = subprocess.run(["curl", "-sS", "--max-time", "120", u], capture_output=True, text=True)
    page = json.loads(p.stdout)
    if not page:
        break
    entries.extend(page)
    if len(page) < 1000:
        break
jpgs = [e for e in entries if e["filename"].endswith(".jpg")]
print("API entries:", len(entries), "| jpgs:", len(jpgs))
stem_to_file = {e["filename"][:-4]: e for e in jpgs}
label_map = {}
for _, row in df.iterrows():
    stem = str(row["File Name"])
    if stem in stem_to_file:
        label_map[stem] = int(row["Ripening Index Classification"])
print("labeled+available images:", len(label_map))
assert len(label_map) > 14000, "unexpectedly few labeled images"
CLASS_NAMES = {1: "Underripe", 2: "Breaking", 3: "Ripe (first stage)",
               4: "Ripe (second stage)", 5: "Overripe"}

## 3. Deterministic stratified subset (60 images/class)

**Purpose /期待結果:** Select exactly 60 images per ripeness class (300 total) with `numpy`'s seeded generator (`seed=42`) over the class-sorted filename list, so the subset is identical on every run. This is the notebook's documented scope reduction (paper: all 14,710 images + augmentation). Expected result: a 300-entry download plan, ~10 MB total (files average ≈33 KB).

In [ ]:
import numpy as np
from collections import Counter

PER_CLASS = 60
rng = np.random.default_rng(42)
selected = []
for cls in sorted(set(label_map.values())):
    stems = sorted(s for s, c in label_map.items() if c == cls)
    pick = rng.choice(len(stems), size=PER_CLASS, replace=False)
    selected.extend((stems[i], cls) for i in sorted(pick))
assert len(selected) == 5 * PER_CLASS
print("selected:", Counter(c for _, c in selected))
total_mb = sum(stem_to_file[s]["size"] for s, _ in selected) / 1e6
print(f"planned download: {total_mb:.1f} MB")

### 3.1 Download the subset images and validate every byte

**Purpose /期待結果:** Download the 300 chosen JPGs from Mendeley via `curl` into `/content/avocado_subset/`, rejecting any non-200 response, and verify each file opens as an RGB image (expected 800×800) with PIL. Expected result: 300 valid images, zero failures, per-class counts printed.

In [ ]:
import os, io, random
from PIL import Image

OUT_DIR = "/content/avocado_subset"
os.makedirs(OUT_DIR, exist_ok=True)
images, labels = [], []
failures = []
for stem, cls in selected:
    fid = stem_to_file[stem]["id"]
    path = os.path.join(OUT_DIR, stem + ".jpg")
    url = f"https://data.mendeley.com/public-files/datasets/3xd9n945v8/files/{fid}/file_downloaded"
    p = subprocess.run(["curl", "-sS", "-L", "--fail", "--max-time", "60",
                        "-o", path, "-w", "%{http_code}", url], capture_output=True, text=True)
    if p.stdout != "200":
        failures.append((stem, p.stdout))
        continue
    img = Image.open(path); img.load()
    if img.mode != "RGB":
        img = img.convert("RGB")
    images.append(img)
    labels.append(cls)
print("downloaded:", len(images), "| failures:", len(failures), failures[:5])
print("sample size:", images[0].size)
assert len(images) == 5 * PER_CLASS, "image download incomplete"
assert all(img.size == (800, 800) for img in images), "unexpected image dimensions"
labels = np.array(labels)
print("class counts:", Counter(labels.tolist()))

### 3.2 Input preview — one real image per class with its true label

**Purpose /期待結果:** Display five original dataset photos (one per ripeness stage) resized for display, each captioned with its true class. This is a real input the reproduction handles, annotated with its Mendeley label — the same role as paper FIGURE 2. Expected result: a 1×5 image grid; visual ripeness progression from green/firm (Underripe) to spotted dark skin (Overripe).

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 5, figsize=(18, 4.2))
shown = set()
idx = 0
for pos, cls in enumerate(sorted(CLASS_NAMES)):
    j = idx
    while labels[j] != cls:
        j += 1
    idx = j + 1
    axes[pos].imshow(images[j].resize((256, 256)))
    axes[pos].set_title(f"{CLASS_NAMES[cls]}\n(index {cls})", fontsize=11)
    axes[pos].axis("off")
fig.suptitle("Input sample: one real Mendeley photo per ripeness class (true labels)", fontsize=13)
fig.tight_layout()
plt.savefig("/content/avocado_input_preview.png", dpi=100, bbox_inches="tight")
plt.show()
print("saved /content/avocado_input_preview.png")

## 4. Preprocessing and feature extraction (author's exact code)

**Purpose /期待結果:** Resize every image 800×800 → 128×128 exactly as the paper's preprocessing section and the author's `Feature Extraction.ipynb` (`Image.resize`), then extract the paper's 8,897-dimensional feature vector per image with the author's function copied verbatim from the pinned commit: HSV histograms (180+256+256 bins) + 9 color moments = 701; GLCM 6 properties × 4 distances × 4 angles = 96; HOG (9 orientations, 8×8 cells, 2×2 blocks, L2-Hys) = 8,100. Expected result: feature matrix `(300, 8897)` — the probe already verified this exact length for one image.

In [ ]:
# Author's extract_hsv_glcm_hog_features, copied verbatim from
# Feature Extraction.ipynb (commit 2d6ab366747a0ae51f394e8789c5106c202fbdc6),
# except that images arrive as PIL objects instead of files read from a local folder.
import cv2
from skimage.feature import graycomatrix, graycoprops, hog
from scipy.stats import skew

def extract_hsv_glcm_hog_features(image):
    image_uint8 = image.astype(np.uint8)
    hsv_image = cv2.cvtColor(image_uint8, cv2.COLOR_RGB2HSV)
    h_hist = cv2.calcHist([hsv_image], [0], None, [180], [0, 180]).flatten()
    s_hist = cv2.calcHist([hsv_image], [1], None, [256], [0, 256]).flatten()
    v_hist = cv2.calcHist([hsv_image], [2], None, [256], [0, 256]).flatten()
    h_hist /= h_hist.sum(); s_hist /= s_hist.sum(); v_hist /= v_hist.sum()
    h_channel = hsv_image[:, :, 0]
    h_mean, h_std, h_skew = np.mean(h_channel), np.std(h_channel), skew(h_channel.flatten())
    s_channel = hsv_image[:, :, 1]
    s_mean, s_std, s_skew = np.mean(s_channel), np.std(s_channel), skew(s_channel.flatten())
    v_channel = hsv_image[:, :, 2]
    v_mean, v_std, v_skew = np.mean(v_channel), np.std(v_channel), skew(v_channel.flatten())
    hsv_features = np.concatenate([h_hist, s_hist, v_hist,
                                   [h_mean, h_std, h_skew],
                                   [s_mean, s_std, s_skew],
                                   [v_mean, v_std, v_skew]])
    gray_image = cv2.cvtColor(image_uint8, cv2.COLOR_RGB2GRAY)
    glcm = graycomatrix(gray_image, distances=[1, 2, 3, 4],
                        angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                        levels=256, symmetric=True, normed=True)
    contrast = graycoprops(glcm, 'contrast').flatten()
    dissimilarity = graycoprops(glcm, 'dissimilarity').flatten()
    homogeneity = graycoprops(glcm, 'homogeneity').flatten()
    asm = graycoprops(glcm, 'ASM').flatten()
    energy = graycoprops(glcm, 'energy').flatten()
    correlation = graycoprops(glcm, 'correlation').flatten()
    hog_features = hog(gray_image, orientations=9, pixels_per_cell=(8, 8),
                       cells_per_block=(2, 2), block_norm='L2-Hys', visualize=False)
    return np.concatenate([hsv_features, contrast, dissimilarity, homogeneity,
                           asm, energy, correlation, hog_features])

features = np.array([extract_hsv_glcm_hog_features(np.asarray(im.resize((128, 128))))
                     for im in images])
print("feature matrix:", features.shape, "(paper/author: 8897 features)")
assert features.shape == (300, 8897)

### 4.1 Stratified 70:15:15 split — author's exact split code

**Purpose /期待結果:** The author's `Feature Extraction.ipynb` splits with `train_test_split(test_size=0.3, random_state=42, stratify=...)`, then halves the 30% temp pool into validation and test. Applied to 300 images this yields **210 train / 45 val / 45 test** (15 per class in val and test), preserving the paper's 70:15:15 protocol at subset scale. Note: the paper's full-scale run also geometrically augments the training pool to 2,500 images/class — omitted here (documented scope reduction).

In [ ]:
from sklearn.model_selection import train_test_split

train_images_f, temp_f, train_labels, temp_labels = train_test_split(
    features, labels, test_size=0.3, random_state=42, stratify=labels)
val_f, test_f, val_labels, test_labels = train_test_split(
    temp_f, temp_labels, test_size=0.5, random_state=42, stratify=temp_labels)
print("train/val/test:", train_images_f.shape, val_f.shape, test_f.shape)
from collections import Counter
print("train classes:", sorted(Counter(train_labels.tolist()).items()))
print("val classes:  ", sorted(Counter(val_labels.tolist()).items()))
print("test classes: ", sorted(Counter(test_labels.tolist()).items()))

### 4.2 Standardize features and reduce to 150 PCA components

**Purpose /期待結果:** Standardize with `StandardScaler` fitted **on the training split only** (author's cell 10), then project to 150 components with `PCA(n_components=150, random_state=42)` fitted on train only (author's cell 12) — the paper's dimensionality-reduction step (8,897 → 150). Expected result: train/val/test matrices of shape (210, 150), (45, 150), (45, 150); printed cumulative explained variance (the paper does not report it; expect the top 150 components to capture roughly 90%+ of the variance of hand-crafted features).

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

scaler = StandardScaler()
train_std = scaler.fit_transform(train_images_f)
val_std = scaler.transform(val_f)
test_std = scaler.transform(test_f)

pca = PCA(n_components=150, random_state=42)
X_train_pca = pca.fit_transform(train_std)
X_val_pca = pca.transform(val_std)
X_test_pca = pca.transform(test_std)
print("PCA shapes:", X_train_pca.shape, X_val_pca.shape, X_test_pca.shape)
print(f"cumulative explained variance (150 comps): {pca.explained_variance_ratio_.sum():.3f}")

## 5. Baseline: default RBF SVM on the PCA features

**Purpose /期待結果:** Train the paper's baseline — `SVC(kernel="rbf")` with scikit-learn defaults, exactly as the author's `PCA+SVM.ipynb` cell 2 — on the PCA features and evaluate on the held-out test split with accuracy/precision/recall/F1 (weighted), the paper's metric set. Expected result: weighted accuracy in a broad 60–90% band (subset scale; the paper reports 81.64% at full scale) — this is the number MFO must beat.

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

model_rbf = SVC(kernel="rbf")
model_rbf.fit(X_train_pca, train_labels)
y_pred_rbf = model_rbf.predict(X_test_pca)

baseline_metrics = {
    "accuracy": accuracy_score(test_labels, y_pred_rbf),
    "precision": precision_score(test_labels, y_pred_rbf, average="weighted"),
    "recall": recall_score(test_labels, y_pred_rbf, average="weighted"),
    "f1": f1_score(test_labels, y_pred_rbf, average="weighted"),
}
for k, v in baseline_metrics.items():
    print(f"baseline RBF {k}: {v*100:.2f}")
print(classification_report(test_labels, y_pred_rbf,
      target_names=[CLASS_NAMES[c] for c in sorted(CLASS_NAMES)]))

## 6. MFO-tuned RBF SVM (author's formulation, niapy)

**Purpose /期待結果:** Reproduce the author's `MFO_RBF.ipynb` cell 3: a two-dimensional maximization problem where x = (log₁₀ C, log₁₀ γ), the RBF-SVM's validation accuracy is the fitness, and niapy's `MothFlameOptimizer(population_size=50)` runs `max_iters=15` (750 evaluations) with the author's bounds — `lower=[-4, -4], upper=[3, 1]`, i.e. C ∈ [1e-4, 1e3], γ ∈ [1e-4, 10]. (The paper text states C up to 1e4; the pinned code uses 1e3 — we follow the code and note the discrepancy. The paper's own optimum C=8.0789 is inside both ranges.) The per-evaluation printouts of the author's `SVMProblem` are reduced to a silent history list; everything else is verbatim. Expected result: best (C, γ) printed with best validation accuracy ≥ baseline-style defaults; several minutes at most on CPU.

In [ ]:
from niapy.task import Task, OptimizationType
from niapy.problems import Problem
from niapy.algorithms.basic import MothFlameOptimizer
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC

class SVMProblem(Problem):
    # Verbatim from MFO_RBF.ipynb (commit 2d6ab36), except the author's per-evaluation
    # print() calls are replaced by a silent history list.
    def __init__(self, X_train, y_train, X_val, y_val):
        super().__init__(dimension=2, lower=[-4, -4], upper=[3, 1])
        self.X_train = X_train
        self.y_train = y_train
        self.X_val = X_val
        self.y_val = y_val
        self.top_solutions = [{"C": 0.0, "gamma": 0.0, "Accuracy": 0.0} for _ in range(10)]
        self.history = []

    def _evaluate(self, x):
        C = 10**x[0]
        gamma = 10**x[1]
        model = SVC(C=C, gamma=gamma, kernel="rbf", random_state=42)
        model.fit(self.X_train, self.y_train)
        predictions = model.predict(self.X_val)
        accuracy = accuracy_score(self.y_val, predictions)
        self.history.append((C, gamma, accuracy))
        new_solution = {"C": C, "gamma": gamma, "Accuracy": accuracy}
        status = True
        for solution in self.top_solutions:
            solution_accuracy = round(solution["Accuracy"], 4)
            new_solution_accuracy = round(accuracy, 4)
            if solution_accuracy == new_solution_accuracy:
                if abs(new_solution["C"] - solution["C"]) < 1 and abs(new_solution["gamma"] - solution["gamma"]) < 1:
                    status = False
                    break
        if status:
            self.top_solutions.append(new_solution)
            self.top_solutions = sorted(self.top_solutions, key=lambda x: x["Accuracy"], reverse=True)[:10]
        return accuracy

problem = SVMProblem(X_train_pca, train_labels, X_val_pca, val_labels)
task = Task(problem, max_iters=15, optimization_type=OptimizationType.MAXIMIZATION)
algorithm = MothFlameOptimizer(population_size=50)
best_params, best_accuracy = algorithm.run(task)
print("best x (log10 C, log10 gamma):", best_params)
print("best validation accuracy:", best_accuracy)
print("evaluations:", len(problem.history))
print("best by validation accuracy:", max(problem.history, key=lambda h: h[2]))

### 6.1 Evaluate the MFO-tuned model on the test split

**Purpose /期待結果:** Retrain an RBF-SVM with the MFO-found (C, γ) — the author's cell 6 does exactly this with the paper's full-scale optimum `C=8.078874923325557, gamma=0.0001` — and compute the paper's four weighted metrics plus a seaborn confusion matrix on the held-out test split. Expected result: test accuracy ≥ the default-RBF baseline from section 5 (the paper's headline claim), with a confusion matrix PNG showing most mass on the diagonal.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

best_C, best_gamma, best_val = max(problem.history, key=lambda h: h[2])
print(f"MFO optimum: C={best_C:.6f}, gamma={best_gamma:.6f} (val acc {best_val:.4f})")
print(f"paper full-scale optimum: C=8.078874923325557, gamma=0.0001")

model_mfo = SVC(kernel="rbf", C=best_C, gamma=best_gamma, random_state=42)
model_mfo.fit(X_train_pca, train_labels)
y_pred_mfo = model_mfo.predict(X_test_pca)

mfo_metrics = {
    "accuracy": accuracy_score(test_labels, y_pred_mfo),
    "precision": precision_score(test_labels, y_pred_mfo, average="weighted"),
    "recall": recall_score(test_labels, y_pred_mfo, average="weighted"),
    "f1": f1_score(test_labels, y_pred_mfo, average="weighted"),
}
for k, v in mfo_metrics.items():
    print(f"MFO-RBF {k}: {v*100:.2f}")
print(classification_report(test_labels, y_pred_mfo,
      target_names=[CLASS_NAMES[c] for c in sorted(CLASS_NAMES)]))

cm = confusion_matrix(test_labels, y_pred_mfo)
fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=[CLASS_NAMES[c] for c in sorted(CLASS_NAMES)],
            yticklabels=[CLASS_NAMES[c] for c in sorted(CLASS_NAMES)], ax=ax)
ax.set_xlabel("Predicted label"); ax.set_ylabel("True label")
ax.set_title(f"Confusion matrix — MFO-RBF on 300-image subset (C={best_C:.3f}, gamma={best_gamma:.4f})")
fig.tight_layout()
plt.savefig("/content/avocado_mfo_confusion_matrix.png", dpi=100)
plt.show()
print("saved /content/avocado_mfo_confusion_matrix.png")

## 7. Baseline vs MFO comparison and paper reference

**Purpose /期待結果:** Print the metric comparison side by side and state whether MFO improved over the default RBF baseline **on this subset**, next to the paper's full-scale reference values (MFO-RBF 82.55% vs baseline RBF 81.60% test accuracy; paper TABLE 5/6). Expected result: a small table; the subset numbers will not equal the paper's — differences are expected from (a) 300 instead of 14,710 images, (b) no geometric augmentation, (c) different random draws — but the qualitative claim (MFO-tuned hyperparameters ≥ defaults) is what this demonstration tests.

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "subset (this notebook)": [baseline_metrics["accuracy"], mfo_metrics["accuracy"]],
    "paper (full data, augmented)": [0.8160, 0.8255],
}, index=["Baseline RBF (default)", "MFO-tuned RBF"])
print((comparison * 100).round(2).to_string())
delta = mfo_metrics["accuracy"] - baseline_metrics["accuracy"]
print(f"\nMFO improvement over baseline on subset: {delta*100:+.2f} percentage points")
print("PASS: MFO >= baseline" if delta >= 0 else "NOTE: MFO did not beat the baseline on this subset")

## 8. Provenance, runtime and limitations

**Purpose /期待結果:** Record the exact pinned sources and the adaptation scope so the session is self-documenting.

- Paper: DOI 10.35882/jeeemi.v7i2.652 (JEEEMI 7(2) 2025, CC BY-SA 4.0).
- Author code: `github.com/kemalcrisannaufal/MFO-SVM-for-Avocado-Ripeness-Classification` @ `2d6ab366747a0ae51f394e8789c5106c202fbdc6` — `Feature Extraction.ipynb` cells 0/1/8/9/10/12 and `MFO_RBF.ipynb` cell 3 are followed verbatim except for the documented changes (PIL input, silent history, subset).
- Dataset: Mendeley 10.17632/3xd9n945v8.1 (CC BY 4.0); XLSX SHA-256 verified; all 300 images downloaded to `/content` inside this session.
- **Limitations /制限:** subset scale (300 of 14,710 images), no geometric augmentation, class-1..5 labels taken from the XLSX `Ripening Index Classification` column (the author's local loader derived labels from renamed files, which the public dataset does not provide). Absolute accuracies are not comparable to the paper's tables; only the baseline-vs-MFO comparison on the identical subset is meaningful.

In [ ]:
import platform, sklearn, skimage, cv2, niapy
print("python:", platform.python_version())
print("numpy", numpy.__version__, "| sklearn", sklearn.__version__,
      "| skimage", skimage.__version__, "| cv2", cv2.__version__,
      "| niapy", im.version("niapy"), "| pandas", pd.__version__)
print("author repo:", f"https://github.com/{REPO}/tree/{COMMIT}")
print("dataset:", "https://data.mendeley.com/datasets/3xd9n945v8/1 (CC BY 4.0)")
print("paper: https://doi.org/10.35882/jeeemi.v7i2.652 (CC BY-SA 4.0)")